<a href="https://colab.research.google.com/github/IoNiCx1/MNIST_numbers/blob/main/ProjectMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

 
import pickle,os,warnings
warnings.filterwarnings('ignore')

print(tf.__version__)
print("GPU ->",tf.config.list_physical_devices('GPU'))

2026-07-25 09:49:00.936184: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-25 09:49:01.055266: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-25 09:49:01.061881: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib/wsl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/python3.12/

2.10.0
GPU -> []


2026-07-25 09:49:03.872226: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-07-25 09:49:03.873087: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib/wsl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib:/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib:/usr/lib/wsl/lib:/usr/local/lib/pyt

In [ ]:
(x_train_full,y_train_full),(x_test,y_test) = keras.datasets.mnist.load_data()
print("Train images shape")

In [ ]:
print("Train images shape:",x_train_full.shape)
print("Train labels shape:",y_train_full.shape)
print("Test images shape:",x_test.shape)
print("Test labels shape:",y_test.shape)


In [ ]:
plt.figure(figsize=(10,4))
for i in range(10):
  plt.subplot(2,5, i+1)
  plt.imshow(x_train_full[i],cmap='gray')
  plt.title(f"Label:{y_train_full[i]}")
  plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
print("Any NaNs in train images?:",np.isnan(x_train_full).any())
print("Any NaNs in test images? :",np.isnan(x_test).any())

flat_train = x_train_full.reshape(x_train_full.shape[0],-1)
n_duplicates = len(flat_train)-len(np.unique(flat_train,axis = 0))
print("Number of Duplicate images in training set:",n_duplicates)

In [ ]:
unique,counts = np.unique(y_train_full,return_counts=True)
class_dist = pd.Series(counts, index = unique)
print(class_dist)
plt.figure(figsize=(7,4))
sns.barplot(x=class_dist.index,y = class_dist.values,palette='viridis')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Class Distribution in training Set')
plt.show()

In [ ]:
print("Min and Max pixel values are : [",x_train_full.min(),",",x_train_full.max(),"]")
x_train_flat = x_train_full.reshape(x_train_full.shape[0],-1).astype('float32')
x_test_flat = x_test.reshape(x_test.shape[0],-1).astype('float32')

print("Flattened train shape:",x_train_flat.shape,"and test shape:",x_test_flat.shape)

*with this the data handling sect is done.*

***Feature Encoding, Scaling and TTS***

In [ ]:
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder
from sklearn.model_selection import train_test_split



# # this cell is for testing (i am testing the encoder because i forgor)
# kl = pd.DataFrame({'Color':['Red','Blue','Yellow']})
# encd = OneHotEncoder(sparse_output=False)
# model_test= encd.fit_transform(kl[['Color']])
# encd_kl= pd.DataFrame(model_test,columns=encd.get_feature_names_out(['Color']) )
# print(encd_kl) # Nice

scalar = MinMaxScaler()
x_train_scalar = scalar.fit_transform(x_train_flat)
x_test_scalar = scalar.transform(x_test_flat)



In [ ]:
encoder = OneHotEncoder(sparse_output=False)
y_train_encoded = encoder.fit_transform(y_train_full.reshape(-1,1))
y_test_encoded = encoder.fit_transform(y_test.reshape(-1,1))

print("Encoded label shape:",y_train_encoded.shape,"\nClassses learned by encoder:",encoder.categories_)

In [ ]:
x_train,x_val,y_train,y_val = train_test_split(
    x_train_scalar,y_train_encoded,
    test_size = 0.15,
    random_state= 42,
    stratify=y_train_full
)

print("Train:",x_train.shape,y_train.shape,"\nVal:",x_val.shape,y_val.shape,"\nTest:\n",x_test_scalar,y_test_encoded.shape)

In [ ]:
from tensorflow._api.v2.config import optimizer
def build_ann_model(input_dim = 784,num_classes= 10):
  model = keras.Sequential([
      layers.Input(shape=(input_dim,)),
      layers.Dense(256,activation = 'relu'),
      layers.Dropout(0.2),
      layers.Dense(128,activation = 'relu'),
      layers.Dropout(0.2),
      layers.Dense(64,activation = 'relu'),
      layers.Dense(num_classes,activation='softmax')
  ])
  model.compile(
      optimizer= 'adam',
      loss = 'categorical_crossentropy',
      metrics = ['accuracy']
      )
  return model

In [ ]:
ann_model = build_ann_model()
ann_model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor = 'val_loss',patience = 5,restore_best_weights = True
) # a function for early stop

In [ ]:
history = ann_model.fit(
    x_train,y_train,
    validation_data= (x_val,y_val),
    epochs = 50,
    batch_size = 128,
    callbacks=[early_stop],
    verbose = 1
)

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history.history['loss'],label = 'train_loss')
ax[0].plot(history.history['val_loss'],label= 'val_loss')
ax[0].set_title('loss');ax[0].set_xlabel('Epoch');ax[0].legend()

ax[1].plot(history.history['accuracy'],label = 'train_acc')
ax[1].plot(history.history['val_accuracy'],label ='val_acc')
ax[1].set_title('Accuracy');ax[1].set_xlabel('Epoch');ax[1].legend()
plt.show()